In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import os

def get_parent(path, levels=1):
    for _ in range(levels):
        path = os.path.dirname(path)
    return path

root_0 = get_parent(os.getcwd(), 3)
root_0

dbox_root = rf"{root_0}\sa_fires\proj_bureaucrats_farms"
shell_root = "/scratch/gpfs/ar8787/groupdata2/sa_fires/proj_bureaucrats_farms"
root = dbox_root

In [ ]:
df = pl.read_csv(fr"{root}/data_output/intermediate/0_master_merge_data_gen.csv", ignore_errors = True)
df = df.with_columns([
        pl.col("year").cast(pl.Int64),
        pl.col("month").cast(pl.Int64)
    ])

In [ ]:
#Election information

myneta = pd.read_stata(f"{root}/data_output/intermediate/panel_data_election_year.dta")
myneta = pl.from_pandas(myneta)
myneta = myneta.with_columns([
        pl.col("ac_uq_id").cast(pl.Int64)
    ])
# df_myneta = df.merge(myneta, on = ['ac_uq_id','year', 'month'], how = 'left', indicator = True)
df_myneta = df.join(
    myneta,
    on=['ac_uq_id', 'year', 'month'],
    how='left'
)


In [ ]:
# myneta_data = df_myneta[['ac_uq_id','year', 'self_profession', 'self_owns_agricultural_assets' ]] \
#         .drop_duplicates()

df_myneta = df_myneta.with_columns(
    pl.when(pl.col("self_profession").is_null() | (pl.col("self_owns_agricultural_assets") == 0))
      .then(None)
      .otherwise(
          pl.col("self_profession").cast(pl.Int64) * pl.col("self_owns_agricultural_assets").cast(pl.Int64)
      )
      .alias("prof_assets")
)

myneta_data = df_myneta.select([
    "ac_uq_id",
    "year",
    "self_profession",
    "self_owns_agricultural_assets"
]).unique()


In [ ]:
# dfwork = df_myneta[['province', 'unique_small_grid_id', 'ac_uq_id','year', 'month', 'count_x', 'downup_ac', 
#                  'self_profession', 'self_owns_agricultural_assets', 'prof_assets',
#                 'av_wind_direction', 'wind_speed']].copy()
# dfwork.columns = ['province', 'unique_small_grid_id', 'ac_uq_id','year', 'month', 'count', 'downup_ac', 
#                  'self_prof', 'self_ass_agri', 'prof_assets',
#                 'av_wind_direction', 'wind_speed']

# # Sort values for consistent lagging
# dfwork = dfwork.sort_values(by=['unique_small_grid_id', 'year', 'month'])
# dfwork = dfwork.replace(np.nan, 9999)

# Select columns
dfwork = df_myneta.select([
    "province",
    "unique_small_grid_id",
    "ac_uq_id",
    "year",
    "month",
    "count",
    "downup_ac",
    "self_profession",
    "prof_assets",
    "self_owns_agricultural_assets",
    "av_wind_speed",
    "wind_direction"
])

# Rename columns
dfwork = dfwork.rename({
    "count": "count",
    "self_profession": "self_prof",
    "self_owns_agricultural_assets": "self_ass_agri"
})

# self_prof arrives from the .dta as text ("0" / "1"). Cast the treatment
# columns to integer so later `self_prof < 9999` comparisons work.
# strict=False turns any non-numeric string into null -> filled with 9999 below.
dfwork = dfwork.with_columns([
    pl.col("self_prof").cast(pl.Int64, strict=False),
    pl.col("prof_assets").cast(pl.Int64, strict=False),
    pl.col("self_ass_agri").cast(pl.Int64, strict=False)
])

# Sort
dfwork = dfwork.sort([
    "unique_small_grid_id",
    "year",
    "month"
])

# Replace NaN and null with 9999
dfwork = dfwork.fill_nan(9999).fill_null(9999)

print(dfwork.head())
print(dfwork["self_prof"].value_counts())

In [ ]:
dfwork = dfwork.to_pandas()

In [ ]:
print(dfwork.self_prof.value_counts())
print(dfwork.self_ass_agri.value_counts())

In [ ]:
cohort_def = myneta.to_pandas()[['month_take', 'year_take']].drop_duplicates().reset_index()
cohort_def.columns = ['index', 'month_take', 'year_take']

In [ ]:
mtlist = cohort_def.month_take.astype(int).tolist()
yrlist = cohort_def.year_take.astype(int).tolist()
# stlist = cohort_def.province.tolist()

for yr, mt in zip(yrlist, mtlist):
    # Step 1: Define reference date
    reference_date = pd.Timestamp(year=yr, month=mt, day=1)

    # Step 2: Create 132-month window from -60 to +71
    month_range = pd.date_range(
        start=reference_date - pd.DateOffset(months=60),
        end=reference_date + pd.DateOffset(months=71),
        freq='MS'
    )

    # Step 3: Create DataFrame
    df_window = pd.DataFrame({'date': month_range})
    df_window['relative_month'] = (
        (df_window['date'].dt.year - reference_date.year) * 12 +
        (df_window['date'].dt.month - reference_date.month)
    )

    # Step 4: Create 12-month grouping variable (like panelview window groups)
    df_window['relative_year_bin'] = df_window['relative_month'] // 12  # integer division

    # Optional: extract year and month
    df_window['year'] = df_window['date'].dt.year
    df_window['month'] = df_window['date'].dt.month
    df_window = df_window.query('relative_year_bin>=-5 & relative_year_bin <= 4')

    df_window['post'] = 0
    df_window.loc[ df_window.relative_year_bin >=0, 'post'] = 1
    df_window['cohort'] = f"{yr}_{mt}"
    # df_window['province'] = st

    df_cohort = df_window.merge(dfwork, on = ['year', 'month'], how = 'left' )

    cohort_tr = df_cohort.query( 'self_prof < 9999 & self_ass_agri < 9999')
    treat_def = cohort_tr.groupby(['unique_small_grid_id', 'post'])[['self_prof', 'self_ass_agri']].max() \
                .reset_index().rename(columns = {'self_prof' : 'treat_prof', 
                                                'self_ass_agri' : 'treat_asset', })

    treat_wide = treat_def.pivot(
        index='unique_small_grid_id',
        columns='post',
        values=['treat_prof', 'treat_asset']
    ).reset_index()
    treat_wide.columns = [f'{var}_post{post}' for var, post in treat_wide.columns]

    try:
        control = treat_wide.query('treat_prof_post0 == 0 & treat_prof_post1 == 0')[['unique_small_grid_id_post']]
        control['treat'] = 0

        treat = treat_wide.query('treat_prof_post0 == 0 & treat_prof_post1 == 1')[['unique_small_grid_id_post']]
        treat['treat'] = 1

        obser = pd.concat([control, treat])
        obser.columns = ['unique_small_grid_id', 'treat']

        cohorte = cohort_tr \
                .merge( obser, how = 'right', on = 'unique_small_grid_id')
        cohorte.to_csv(os.path.join(root,'data_output','intermediate','cohorts_anyst_analysis',f'{yr}_{mt}.csv'), index = False )
    except:
        print(f'Error in year {yr} - month {mt}')

### Self Assets

In [ ]:
mtlist = cohort_def.month_take.tolist()
yrlist = cohort_def.year_take.tolist()

for yr, mt in zip(yrlist, mtlist):
    # Step 1: Define reference date
    reference_date = pd.Timestamp(year=yr, month=mt, day=1)

    # Step 2: Create 132-month window from -60 to +71
    month_range = pd.date_range(
        start=reference_date - pd.DateOffset(months=60),
        end=reference_date + pd.DateOffset(months=71),
        freq='MS'
    )

    # Step 3: Create DataFrame
    df_window = pd.DataFrame({'date': month_range})
    df_window['relative_month'] = (
        (df_window['date'].dt.year - reference_date.year) * 12 +
        (df_window['date'].dt.month - reference_date.month)
    )

    # Step 4: Create 12-month grouping variable (like panelview window groups)
    df_window['relative_year_bin'] = df_window['relative_month'] // 12  # integer division

    # Optional: extract year and month
    df_window['year'] = df_window['date'].dt.year
    df_window['month'] = df_window['date'].dt.month
    df_window = df_window.query('relative_year_bin>=-5 & relative_year_bin <= 4')

    df_window['post'] = 0
    df_window.loc[ df_window.relative_year_bin >=0, 'post'] = 1
    df_window['cohort'] = f"{yr}_{mt}"

    df_cohort = df_window.merge(dfwork, on = ['year', 'month'], how = 'left' )

    cohort_tr = df_cohort.query( 'self_prof < 9999 & self_ass_agri < 9999')
    treat_def = cohort_tr.groupby(['unique_small_grid_id', 'post'])[['self_prof', 'self_ass_agri']].max() \
                .reset_index().rename(columns = {'self_prof' : 'treat_prof', 
                                                'self_ass_agri' : 'treat_asset', })

    treat_wide = treat_def.pivot(
        index='unique_small_grid_id',
        columns='post',
        values=['treat_prof', 'treat_asset']
    ).reset_index()
    treat_wide.columns = [f'{var}_post{post}' for var, post in treat_wide.columns]

    columnsdf = treat_wide.columns.tolist()
    if (('treat_asset_post0' in columnsdf)  & ( 'treat_asset_post1'  in columnsdf)):
        control = treat_wide.query('treat_asset_post0 == 0 & treat_asset_post1 == 0')[['unique_small_grid_id_post']]
        control['treat'] = 0

        treat = treat_wide.query('treat_asset_post0 == 0 & treat_asset_post1 == 1')[['unique_small_grid_id_post']]
        treat['treat'] = 1

        obser = pd.concat([control, treat])
        obser.columns = ['unique_small_grid_id', 'treat']

        cohorte = cohort_tr \
                .merge( obser, how = 'right', on = 'unique_small_grid_id')
        cohorte.to_csv(fr'C:\Users\Anzony\Documents\sa_fires/cohortes_assets_anyst/{yr}_{mt}.csv', index = False )

### Self Assets & Self Prof

In [ ]:
mtlist = cohort_def.month_take.tolist()
yrlist = cohort_def.year_take.tolist()

In [ ]:
yr = yrlist[0]
mt = mtlist[0]
print(yr, mt)

In [ ]:

cohort_tr = df_cohort.query( 'self_prof < 9999 & prof_assets < 9999')
treat_def = cohort_tr.groupby(['unique_small_grid_id', 'post'])[['self_prof', 'prof_assets']].max() \
            .reset_index().rename(columns = {'self_prof' : 'treat_prof', 
                                            'prof_assets' : 'treat_asset', })
treat_def['treat'] = treat_def['treat_prof'] * treat_def['treat_asset']
treat_wide = treat_def.pivot(
    index='unique_small_grid_id',
    columns='post',
    values=['treat']
).reset_index()
treat_wide.columns = [f'{var}_post{post}' for var, post in treat_wide.columns]
columnsdf = treat_wide.columns.tolist()

In [ ]:
mtlist = cohort_def.month_take.tolist()
yrlist = cohort_def.year_take.tolist()

for yr, mt in zip(yrlist, mtlist):
    # Step 1: Define reference date
    reference_date = pd.Timestamp(year=yr, month=mt, day=1)

    # Step 2: Create 132-month window from -60 to +71
    month_range = pd.date_range(
        start=reference_date - pd.DateOffset(months=60),
        end=reference_date + pd.DateOffset(months=71),
        freq='MS'
    )

    # Step 3: Create DataFrame
    df_window = pd.DataFrame({'date': month_range})
    df_window['relative_month'] = (
        (df_window['date'].dt.year - reference_date.year) * 12 +
        (df_window['date'].dt.month - reference_date.month)
    )

    # Step 4: Create 12-month grouping variable (like panelview window groups)
    df_window['relative_year_bin'] = df_window['relative_month'] // 12  # integer division

    # Optional: extract year and month
    df_window['year'] = df_window['date'].dt.year
    df_window['month'] = df_window['date'].dt.month
    df_window = df_window.query('relative_year_bin>=-5 & relative_year_bin <= 4')

    df_window['post'] = 0
    df_window.loc[ df_window.relative_year_bin >=0, 'post'] = 1
    df_window['cohort'] = f"{yr}_{mt}"

    df_cohort = df_window.merge(dfwork, on = ['year', 'month'], how = 'left' )

    cohort_tr = df_cohort.query( 'prof_assets < 9999')
    treat_def = cohort_tr.groupby(['unique_small_grid_id', 'post'])[['prof_assets']].max() \
                .reset_index().rename(columns = {'prof_assets' : 'treat_prof_assets'})

    treat_wide = treat_def.pivot(
        index='unique_small_grid_id',
        columns='post',
        values=['treat_prof_assets']
    ).reset_index()
    treat_wide.columns = [f'{var}_post{post}' for var, post in treat_wide.columns]
    columnsdf = treat_wide.columns.tolist()
    if (('treat_prof_assets_post0' in columnsdf)  & ( 'treat_prof_assets_post1'  in columnsdf)):
        control = treat_wide.query('treat_prof_assets_post0 == 0 & treat_prof_assets_post1 == 0')[['unique_small_grid_id_post']]
        control['treat'] = 0

        treat = treat_wide.query('treat_prof_assets_post0 == 0 & treat_prof_assets_post1 == 1')[['unique_small_grid_id_post']]
        treat['treat'] = 1

        obser = pd.concat([control, treat])
        obser.columns = ['unique_small_grid_id', 'treat']

        cohorte = cohort_tr \
                .merge( obser, how = 'right', on = 'unique_small_grid_id')
        cohorte.to_csv(fr'{root}/data_output/intermediate/cohortes_prof_assets_anyst/{yr}_{mt}.csv', index = False )